# 08. 엄격판·교정판 데이터셋 (판정기준서 v1.1 9장, D64, 2026-11-23)

- 엄격판: 증권 CoT에서 제외 처리된 CoT를 뺀 것, 교정 없음. 교정판: 엄격판 + 교정(P7 cot2 범위, D22 메타 연령, ④ 등급 문장).
- 함수: `src/dataset.py` (애매한 경우 처리는 함수 주석). 원문이 들어 있어 `outputs/corrected/`(git 제외)에 저장한다.
- 교정 기록은 `outputs/corrections_log.csv`에 더한다 (원문 없음). 이 노트북을 다시 돌리면 D22·④ 기록은 지우고 다시 쓴다.

데이터: AI Hub(한국지능정보사회진흥원) 사업 결과

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 220)
from src.load_cot import load_cot, load_products
from src.judge import judge
from src.dataset import build
AS_OF = "2026-11-23"
cot = load_cot(); prod = load_products()
j, ctx = judge(cot, prod, as_of=AS_OF)
d4 = pd.read_csv(ROOT / "outputs" / "d4_cause_by_product.csv", encoding="utf-8-sig", keep_default_na=False)
latest = pd.read_csv(ROOT / "outputs" / "latest_grade_check.csv", encoding="utf-8-sig", keep_default_na=False, na_values=[""])
g4 = pd.read_csv(ROOT / "outputs" / "latest_grade_rejudge.csv", encoding="utf-8-sig", keep_default_na=False)
g4_ids = set(g4.loc[g4["G4 태그"] != "", "cot_id"].astype(int))
strict, corr, newlog = build(cot, j, d4, latest, g4_ids, ctx["recs"])
print("증권 CoT:", len(j), "/ 엄격판:", len(strict), "/ 교정판:", len(corr), "/ 제외:", len(j) - len(strict))
print("분할:", strict["split"].value_counts().to_dict())

## 1. 교정 내역

In [ ]:
old = pd.read_csv(ROOT / "outputs" / "corrections_log.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
old = old[~old["위치"].isin(["메타 연령", "answer 위험등급 문장"])]
print("기존 기록(cot2 허용 범위):", len(old), "/ 그중 엄격판에 남은 CoT:", old["cot_id"].astype(int).isin(strict["cot_id"]).sum())
print("새 기록:", newlog["위치"].value_counts().to_dict())
a = newlog[newlog["위치"] == "answer 위험등급 문장"]
print("④ 등급 문장: 상품", len(a), "/ CoT", a["cot_id"].nunique(), "/ 등급 출처:", a["근거"].str.extract(r"\((.*)\)")[0].value_counts().to_dict())
miss = d4[(d4["answer"] != "있음") & d4["cot_id"].isin(corr["cot_id"])].drop_duplicates(["cot_id", "상품명"])
print("등급이 없어 문장을 못 붙인 미기재 상품:", len(miss) - len(a))
s = strict.set_index("cot_id"); c = corr.set_index("cot_id")
for col in ["cot2", "age", "answer"]:
    print(f"교정판에서 바뀐 {col}:", int((s[col].astype(str) != c[col].astype(str)).sum()), "CoT")
log = pd.concat([old, newlog.astype(str)], ignore_index=True)
log.to_csv(ROOT / "outputs" / "corrections_log.csv", index=False, encoding="utf-8-sig")
print("corrections_log 행:", len(log))

## 2. 저장 (`outputs/corrected/`, git 제외)과 태그 분포

In [ ]:
od = ROOT / "outputs" / "corrected"; od.mkdir(exist_ok=True)
cols = ["cot_id", "split", "category", "gender", "age", "query_type", "question", "cot1", "cot2", "cot3", "answer", "product_names", "태그"]
for name, df in [("엄격판", strict), ("교정판", corr)]:
    for sp, g in df.groupby("split"):
        with open(od / f"{name}_{sp}.jsonl", "w", encoding="utf-8") as f:
            for rec in g[cols].to_dict("records"):
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(name, "저장:", df["split"].value_counts().to_dict())
t = strict[["split", "태그"]].assign(태그=strict["태그"].str.split(" | ", regex=False)).explode("태그")
t = t[t["태그"] != ""]
print(pd.crosstab(t["태그"], t["split"], margins=True))
print("태그 없는 CoT:", (strict["태그"] == "").sum())

In [ ]:
from src.report import update_scorecard
R = lambda k, t, v, a, u: {"지표": k, "Training": t, "Validation": v, "전체": a, "단위": u, "출처": "08_정제데이터셋", "근거 결정": "판정기준서 9장, D64", "기준일": AS_OF}
cnt = lambda df, sp: int((df["split"] == sp).sum())
cc = newlog[newlog["위치"] == "answer 위험등급 문장"].merge(corr[["cot_id", "split"]], on="cot_id")
nt = strict[strict["태그"] == ""]
rows = [R("엄격판 CoT", cnt(strict, "Training"), cnt(strict, "Validation"), len(strict), "CoT"),
        R("교정판 CoT", cnt(corr, "Training"), cnt(corr, "Validation"), len(corr), "CoT"),
        R("교정: cot2 허용 범위 (P7)", "", "", int((s["cot2"].astype(str) != c["cot2"].astype(str)).sum()), "CoT"),
        R("교정: 메타 연령 (D22)", "", "", int((s["age"].astype(str) != c["age"].astype(str)).sum()), "CoT"),
        R("교정: ④ 등급 문장 추가 상품", int((cc["split"] == "Training").sum()), int((cc["split"] == "Validation").sum()), len(cc), "추천"),
        R("교정: ④ 등급 문장 추가 CoT", cc.loc[cc["split"] == "Training", "cot_id"].nunique(), cc.loc[cc["split"] == "Validation", "cot_id"].nunique(), cc["cot_id"].nunique(), "CoT"),
        R("④ 미기재인데 등급이 없어 문장 미추가 상품", "", "", len(miss) - len(a), "추천"),
        R("태그 없는 CoT (엄격판)", cnt(nt, "Training"), cnt(nt, "Validation"), len(nt), "CoT")]
update_scorecard(rows, "정제 데이터셋 (08)")
print("scorecard 저장")

---
데이터: AI Hub(한국지능정보사회진흥원) 사업 결과